# `cssd` — release smoke test

Exercises the CSSD (cubic smoothing splines for discontinuous signals) Python API on small synthetic inputs to verify a release is functional. Useful as both:

- A post-publish check after `pip install cssd`.
- A first-time-user introduction to the API.

Each section is self-contained. Run-all should complete in a few seconds.

## 1. Install and import

Uncomment the install line if `cssd` isn't on the active Python.

In [ ]:
%pip install --upgrade cssd matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt
import cssd

rng = np.random.default_rng(0)

# Verify the Rust extension is loaded.
import cssd._cssd_core as _core
print('Rust core loaded from:', _core.__file__)
assert 'site-packages' in _core.__file__, (
    f'cssd imported from local source, not from a wheel: {_core.__file__}'
)

## 2. Smooth-only regime (γ → ∞ ≡ classical smoothing spline)

When the discontinuity penalty γ is set to `np.inf`, the model degenerates to the classical Reinsch smoothing spline — no jumps, just smoothness controlled by `p`.

In [ ]:
n = 60
x = np.linspace(0, 2 * np.pi, n)
y_clean = np.sin(x) + 0.3 * np.cos(3 * x)
y = y_clean + rng.normal(0, 0.15, size=n)

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, p in zip(axes, [0.2, 0.7, 0.99]):
    out = cssd.cssd(x, y, p=p, gamma=np.inf)
    xx = np.linspace(x[0], x[-1], 400)
    yy = out.pp(xx).ravel()
    ax.scatter(x, y, s=12, color='lightgray', label='noisy')
    ax.plot(x, y_clean, '--', color='black', linewidth=0.7, label='truth')
    ax.plot(xx, yy, color='C0', linewidth=2, label=f'p={p}')
    ax.set_title(f'γ=∞, p={p}'); ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

## 3. With a discontinuity

A signal with a real jump. CSSD finds it; classical smoothing splines blur it. We compare γ=∞ (no jumps allowed) vs γ=1.0 (one jump permitted at low cost).

In [ ]:
n = 80
x = np.linspace(0, 1, n)
y_truth = np.where(x < 0.5, np.sin(4 * x), 1.0 + np.sin(4 * x))
y = y_truth + rng.normal(0, 0.08, size=n)

out_no_jump = cssd.cssd(x, y, p=0.99, gamma=np.inf)
out_with_jump = cssd.cssd(x, y, p=0.99, gamma=1.0)

xx = np.linspace(x[0], x[-1], 400)
fig, ax = plt.subplots(figsize=(7, 3))
ax.scatter(x, y, s=12, color='lightgray', label='noisy')
ax.plot(x, y_truth, '--', color='black', linewidth=0.7, label='truth')
ax.plot(xx, out_no_jump.pp(xx).ravel(), color='C1', linewidth=1.5, label='γ=∞')
ax.plot(xx, out_with_jump.pp(xx).ravel(), color='C0', linewidth=2,
        label=f'γ=1.0 (jumps={len(out_with_jump.discont)})')
ax.legend(); ax.set_title('CSSD recovers the discontinuity; γ=∞ blurs it')
plt.tight_layout(); plt.show()

assert len(out_with_jump.discont) == 1, 'expected exactly one discontinuity'

## 4. Multiple discontinuities + heteroscedastic noise

Three segments, varying noise levels. `delta` lets us pass per-point standard deviations to down-weight noisy regions.

In [ ]:
n = 100
x = np.linspace(0, 1, n)
y_truth = np.where(x < 0.33, x,
          np.where(x < 0.66, 1.5 - 0.3 * x, -0.2 + 0.5 * np.sin(8 * x)))
delta = np.where(x < 0.5, 0.05, 0.15)  # left half low-noise, right half high-noise
y = y_truth + rng.normal(0, delta)

out = cssd.cssd(x, y, p=0.95, gamma=0.5, delta=delta)
xx = np.linspace(x[0], x[-1], 500)
yy = out.pp(xx).ravel()

fig, ax = plt.subplots(figsize=(8, 3))
ax.errorbar(x, y, yerr=delta, fmt='.', color='lightgray', capsize=2, markersize=4, label='noisy ± δ')
ax.plot(x, y_truth, '--', color='black', linewidth=0.7, label='truth')
ax.plot(xx, yy, color='C0', linewidth=2, label=f'CSSD (jumps={len(out.discont)})')
for d in out.discont:
    ax.axvline(d, color='C3', linewidth=0.5, alpha=0.6)
ax.legend(); ax.set_title('Heteroscedastic CSSD with discontinuities')
plt.tight_layout(); plt.show()

## 5. FPVI vs PELT pruning give the same answer

FPVI (default, fast) and PELT (alternative) are different DP-acceleration strategies for the same optimisation problem. They must produce identical outputs (down to numerical noise) on every input.

In [ ]:
out_fpvi = cssd.cssd(x, y, p=0.95, gamma=0.5, delta=delta, pruning='FPVI')
out_pelt = cssd.cssd(x, y, p=0.95, gamma=0.5, delta=delta, pruning='PELT')

max_coef_diff = np.max(np.abs(out_fpvi.pp.coefs - out_pelt.pp.coefs))
max_break_diff = np.max(np.abs(out_fpvi.pp.breaks - out_pelt.pp.breaks))
discont_match = np.array_equal(out_fpvi.discont_idx, out_pelt.discont_idx)

print(f'max |coefs_FPVI - coefs_PELT| = {max_coef_diff:.3e}')
print(f'max |breaks_FPVI - breaks_PELT| = {max_break_diff:.3e}')
print(f'discontinuity indices match exactly: {discont_match}')
assert max_coef_diff < 1e-10
assert max_break_diff < 1e-12
assert discont_match

## 6. Automatic (p, γ) selection via cross-validation

`cssd_cv` runs simulated annealing over the (p, γ) plane, scoring each candidate by K-fold CV residual. Useful when you don't know the hyperparameters a priori. The 5-fold default is fast on small signals.

In [ ]:
n_t = 80
x_t = np.linspace(0, 1, n_t)
y_truth_t = np.where(x_t < 0.5, x_t, 1 - x_t)  # tent
y_t = y_truth_t + rng.normal(0, 0.05, size=n_t)

cv_out = cssd.cssd_cv(x_t, y_t, cv_type='random', cv_arg=5, max_time=2.0, seed=0)
p_star, gamma_star = cv_out.p, cv_out.gamma
out_cv = cssd.cssd(x_t, y_t, p=p_star, gamma=gamma_star)

print(f'CV-selected (p, γ) = ({p_star:.3f}, {gamma_star:.3g})')
print(f'discontinuities found: {len(out_cv.discont)}')

xx = np.linspace(x_t[0], x_t[-1], 300)
fig, ax = plt.subplots(figsize=(7, 3))
ax.scatter(x_t, y_t, s=12, color='lightgray', label='noisy')
ax.plot(x_t, y_truth_t, '--', color='black', linewidth=0.7, label='truth')
ax.plot(xx, out_cv.pp(xx).ravel(), color='C0', linewidth=2,
        label=f'CV: p={p_star:.2f}, γ={gamma_star:.2g}')
ax.legend(); ax.set_title('Hyperparameters chosen by 5-fold CV')
plt.tight_layout(); plt.show()

## 7. Summary

In [ ]:
import importlib.metadata
print(f'cssd {importlib.metadata.version("cssd")} — smoke test OK')